# Solve a harmonic oscillator with a quantum Taylor expansion

The initial-value problem uses unit frequency and mass, position one, velocity one, and zero driving. Prepare the eight degree-seven Taylor coefficients, select the eight powers of the two-dimensional evolution generator, and uncompute the encoding. The circuit keeps the position and velocity qubit, the three-bit coefficient register, and the zero-driving selector. The existing bounded lowering can use two additional clean work bits.

The eleven times are 0.0 through 1.0 by 0.1. Read both signed amplitudes conditioned on the two selectors being zero, and multiply by the original coefficient normalization. This retains the velocity zero crossing. The source table used amplitude magnitudes and swapped the energy labels in its plotting code. The native table includes the original magnitudes and uses position squared over two for potential energy and velocity squared over two for kinetic energy.

Run with FlagQuantum, Matplotlib and Jupyter installed. The figure is written under the Notebook working directory.


In [ ]:
from importlib import import_module

import torch

from examples.synthesis.community.paper_implementation_project.quantum_algo_for_solving_linear_differential_equations.harmonic_oscillator.overview.main import (
    OUTPUT,
)
from examples.synthesis.community.paper_implementation_project.quantum_algo_for_solving_linear_differential_equations.harmonic_oscillator.overview.model import (
    GENERATOR,
    GRID,
    INITIAL,
    OFFSET,
    ORDER,
    taylor_encoding,
)
from examples.synthesis.community.paper_implementation_project.quantum_algo_for_solving_linear_differential_equations.harmonic_oscillator.overview.plots import (
    save_comparison,
)
from examples.synthesis.community.paper_implementation_project.quantum_algo_for_solving_linear_differential_equations.harmonic_oscillator.overview.quantum import (
    build_program,
)
from examples.synthesis.community.paper_implementation_project.quantum_algo_for_solving_linear_differential_equations.harmonic_oscillator.overview.reference import (
    direct_taylor,
)
from examples.synthesis.community.paper_implementation_project.quantum_algo_for_solving_linear_differential_equations.harmonic_oscillator.overview.workflow import (
    run_workflow,
    show_observations,
)


## Initial-value problem

In [ ]:
print("Initial state:", INITIAL, "Offset:", OFFSET, "Taylor degree:", ORDER)


## Compute the Taylor and offset branches

In [ ]:
encoding = taylor_encoding(1.0)
print("Fresh time-one Taylor weights:", encoding.weights)


In [ ]:
assert OFFSET == (0.0, 0.0)
zero_offset_encoding = torch.eye(8, dtype=torch.float64)
print("Offset encoding width:", zero_offset_encoding.shape[0])


In [ ]:
branch_selector = (1.0, -1.0)
print("Branch selector diagonal:", branch_selector)


## Encoding, controlled evolution and decoding

In [ ]:
encoded = torch.tensor(encoding.householder, dtype=torch.float64)
print("Normalized coefficient amplitudes:", encoded[:, 0].tolist())


In [ ]:
generator = torch.tensor(GENERATOR, dtype=torch.float64)
powers = [torch.linalg.matrix_power(generator, k) for k in range(ORDER + 1)]
print("Selected evolution matrices:", len(powers))


In [ ]:
torch.testing.assert_close(
    encoded.T @ encoded, zero_offset_encoding, atol=1e-12, rtol=0
)


In [ ]:
program = build_program(encoding)
print("Named registers:", tuple(program.registers))
print("Native resources:", program.report["resources"])


## Read the signed solution at every source time

In [ ]:
observations = run_workflow()
assert tuple(row.time for row in observations) == GRID
for row in observations:
    reference = direct_taylor(row.time).tolist()
    print(
        f"t={row.time:.1f}: signed ({row.position:.9f}, {row.velocity:.9f}), Taylor {reference}"
    )


In [ ]:
image = import_module("IPython.display")
image.display(image.Image(filename=str(save_comparison(observations, OUTPUT))))


In [ ]:
show_observations(observations)
